# 🛍️ ShopUNow — Multi-User Conversational Agentic RAG System (LangGraph)

This notebook implements a **multi-user conversational Agentic RAG pipeline** for ShopUNow's internal knowledge base using **LangGraph** with per-user memory.

## Learning Objectives
In this notebook, you will learn:
1. **Department vector stores** - load the seven per-department Chroma stores (building any that are missing)
2. **Graph state and nodes** - classify, retrieve, generate and grade groundedness as LangGraph nodes
3. **Bounded self-correction with memory** - wire a capped regenerate loop and compile the graph with a per-user `MemorySaver`
4. **Multi-user conversations** - hold isolated, follow-up-aware conversations for two users
5. **Negative tests** - check behaviour on out-of-scope, gibberish and cross-user-leakage inputs
6. **Persistent memory (optional)** - swap `MemorySaver` for `SqliteSaver`

## Key Features
- **Conversation Memory**: Each user gets isolated conversation history via LangGraph's checkpointer
- **Context-Aware Classification**: Follow-up queries are classified using conversation context
- **Context-Aware Generation**: Responses reference prior exchanges for natural conversation flow
- **Hallucination Grading**: Every response is verified against retrieved documents

## Architecture

```
User Query + Conversation History
    │
    ▼
┌──────────────────┐
│  Categorize Query │  ← LLM classifies → one of 7 departments
│  (context-aware)  │    (uses conversation history for follow-ups)
└────────┬─────────┘
         ▼
┌──────────────────┐
│   Retrieve Docs   │  ← similarity search on department's ChromaDB
└────────┬─────────┘
         ▼
┌──────────────────┐
│ Generate Response │  ← LLM synthesizes answer from context + history
│  (history-aware)  │
└────────┬─────────┘
         ▼
┌──────────────────┐
│Hallucination Check│  ← LLM grades if response is grounded in docs
└────────┬─────────┘
    ┌────┴────┐
    ▼         ▼
┌────────┐  [Regenerate]
│Finalize│   (max 1 retry)
└────┬───┘
     ▼
   [END]

Memory Layer: MemorySaver (per-thread conversation isolation)
```

## Where this fits
`01_create_vector_databases.ipynb` embedded each department's QA pairs into its own Chroma store under `vectorstores/`.
This notebook routes every question to one of those stores and wraps retrieval in a LangGraph agent with memory and a hallucination check.
If a store is missing, Part 1 builds it from `data/` the same way, so this notebook also runs on its own.

## Prerequisites
- LangGraph state graphs, conditional edges and checkpointers (repository 04) and RAG basics (repository 05); `01_create_vector_databases.ipynb` as reading
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` (chat model `gpt-4o-mini`) and `DATABRICKS_HOST` / `DATABRICKS_TOKEN` (embeddings via the project's `helpers.get_embeddings()`) in this project's `.env`

---
## 📦 Step 1: Environment Setup

Load the keys from this project's `.env`, create the chat model through the course's LLM factory, then import LangGraph, Chroma and the project's embedding factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY (chat) and DATABRICKS_HOST / DATABRICKS_TOKEN (embeddings) from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model for classification, generation and grading
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# Imports
# ============================================================================

import json
import sys
from pathlib import Path
from typing import TypedDict, Literal, Annotated

from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, trim_messages

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# Make the project folder (where helpers/ lives) importable. Jupyter's working
# directory is the notebook's folder, which is the project folder here.
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from helpers.utils import get_embeddings

---
## 🗂️ Part 1: Load the Department Vector Stores

The agent retrieves from one Chroma store per department. `load_vectorstore` reopens a persisted store;
if a store is empty because `01_create_vector_databases.ipynb` has not been run, `build_vectorstore_from_data`
builds it from `data/<department>_data.json` with the same embedding model and collection name.

In [ ]:
# ============================================================================
# Configuration & Helpers
# ============================================================================

VECTORSTORE_DIR = Path("vectorstores")
DATA_DIR = Path("data")

DEPARTMENTS = [
    "hr",
    "it_support",
    "facilities_admin",
    "customer_service",
    "product_sales",
    "billing_payments",
    "shipping_delivery",
]

# Must match the collection naming used by 01_create_vector_databases.ipynb
def collection_name_for(dept: str) -> str:
    return f"dept_{dept}"


# Embedding model from the project's helpers factory - must be the one that built the stores
embeddings = get_embeddings()

# Maximum approximate token budget for conversation history in prompts
MAX_HISTORY_TOKENS = 4000


def load_vectorstore(dept: str) -> Chroma:
    """Load a persistent ChromaDB vectorstore for the given department."""
    persist_dir = str(VECTORSTORE_DIR / dept)
    return Chroma(
        persist_directory=persist_dir,
        embedding_function=embeddings,
        collection_name=collection_name_for(dept),
    )


def build_vectorstore_from_data(dept: str) -> Chroma:
    """Build a department's store from data/<dept>_data.json (same steps as 01_create_vector_databases.ipynb)."""
    records = json.loads((DATA_DIR / f"{dept}_data.json").read_text())
    return Chroma.from_texts(
        texts=[r["doc"] for r in records],
        metadatas=[{"category": r["category"]} for r in records],
        embedding=embeddings,
        persist_directory=str(VECTORSTORE_DIR / dept),
        collection_name=collection_name_for(dept),
    )


# Quick sanity check - build any store that is missing or empty
for dept in DEPARTMENTS:
    vs = load_vectorstore(dept)
    if vs._collection.count() == 0:
        vs = build_vectorstore_from_data(dept)
        print(f"{dept:<22} — store was missing: built it from {DATA_DIR}/")
    print(f"{dept:<22} — {vs._collection.count()} vectors")

---
## 🧩 Part 2: Graph State and Nodes

The graph passes one `AgentState` dictionary between five nodes. `messages` uses the `add_messages` reducer,
so each finished turn is appended to the user's history; every other field is overwritten per question.

### Key Concepts:
- **Structured output**: the classifier and the grader return Pydantic models, so routing never parses free text
- **History-aware prompts**: recent turns are trimmed and formatted into the classifier and generator prompts

### 2.1 🧾 State Definition

In [ ]:
# ============================================================================
# State Definition
# ============================================================================

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]  # conversation history (per-thread)
    query: str                        # original user question
    department: str | None            # classified department key
    retrieved_docs: str | None        # formatted retrieved context
    response: str | None              # generated answer
    hallucination_grade: str | None   # "grounded" or "not_grounded"
    retries: int                      # regeneration counter (cap at 1)

### 2.2 🧭 Node 1 — Categorize Query

The classifier sees the last few turns, so a follow-up such as "What about refunds?" stays in the department of the previous question.
Its output is constrained to the seven department keys.

In [ ]:
# ============================================================================
# Node 1 — Categorize Query (Conversation-Aware)
# ============================================================================

DepartmentKey = Literal[
    "hr",
    "it_support",
    "facilities_admin",
    "customer_service",
    "product_sales",
    "billing_payments",
    "shipping_delivery",
]


class QueryClassification(BaseModel):
    """Classify a user query into one of the ShopUNow departments."""
    department: DepartmentKey = Field(
        description="The department best suited to answer this query."
    )


classification_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a query classifier for ShopUNow, an e-commerce company.
Classify the user's query into exactly one of these departments:

- hr: Leave policy, salary, appraisals, onboarding, employee benefits
- it_support: Password resets, VPN, software installs, hardware issues, email
- facilities_admin: Meeting rooms, office supplies, parking, building access, maintenance
- customer_service: Complaints, returns, account issues, order problems
- product_sales: Product info, discounts, availability, recommendations, pricing
- billing_payments: Invoices, refunds, payment methods, transaction issues
- shipping_delivery: Tracking, delivery times, shipping costs, address changes

If conversation history is provided, use it to understand what the user is referring to
(e.g., "What about refunds?" after a billing question should stay in billing_payments).

Return ONLY the department key."""),
    ("human", """Conversation history:
{history}

Current query: {query}"""),
])

classification_chain = classification_prompt | llm.with_structured_output(QueryClassification)


def _format_history(messages: list, max_turns: int = 3) -> str:
    """Format recent conversation history for prompt injection."""
    if not messages:
        return "(no prior conversation)"
    recent = messages[-(max_turns * 2):]
    lines = []
    for msg in recent:
        role = "User" if isinstance(msg, HumanMessage) else "Assistant"
        content = msg.content[:200] + "..." if len(msg.content) > 200 else msg.content
        lines.append(f"{role}: {content}")
    return "\n".join(lines) if lines else "(no prior conversation)"


def categorize_query(state: AgentState) -> dict:
    """Classify the user query into a department, using conversation context."""
    history = _format_history(state.get("messages", []))
    result = classification_chain.invoke({
        "query": state["query"],
        "history": history,
    })
    print(f"[categorize_query] → {result.department}")
    return {"department": result.department}

### 2.3 🔎 Node 2 — Retrieve Docs

Only the classified department's store is searched (top 3 QA pairs).

In [ ]:
# ============================================================================
# Node 2 — Retrieve Docs
# ============================================================================

def retrieve_docs(state: AgentState) -> dict:
    """Load the department vectorstore and run similarity search."""
    dept = state["department"]
    vs = load_vectorstore(dept)
    results = vs.similarity_search(state["query"], k=3)

    formatted = "\n\n---\n\n".join(
        f"[Doc {i+1}] {doc.page_content}" for i, doc in enumerate(results)
    )
    print(f"[retrieve_docs] Retrieved {len(results)} docs from '{dept}'")
    return {"retrieved_docs": formatted}

### 2.4 ✍️ Node 3 — Generate Response

The generator answers only from the retrieved context, with the trimmed conversation history for follow-ups.

In [ ]:
# ============================================================================
# Node 3 — Generate Response (History-Aware)
# ============================================================================

generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful ShopUNow assistant.
Use ONLY the provided context to answer the user's question.
If the context does not contain enough information, say so clearly.
Do not make up information.

If conversation history is provided, use it to give contextual, conversational answers.
Reference prior exchanges naturally when relevant."""),
    ("human", """Conversation history:
{history}

Question: {query}

Context:
{context}

Answer:"""),
])

generation_chain = generation_prompt | llm


def generate_response(state: AgentState) -> dict:
    """Generate an answer from retrieved context + conversation history."""
    messages = state.get("messages", [])
    trimmed = trim_messages(
        messages,
        max_tokens=MAX_HISTORY_TOKENS,
        strategy="last",
        token_counter=lambda msgs: sum(len(m.content) // 4 for m in msgs),
        allow_partial=False,
    )
    history = _format_history(trimmed)

    result = generation_chain.invoke({
        "query": state["query"],
        "context": state["retrieved_docs"],
        "history": history,
    })
    print(f"[generate_response] Response generated ({len(result.content)} chars)")
    return {"response": result.content}

### 2.5 🛡️ Node 4 — Hallucination Check and Router

The grader labels the answer `grounded` or `not_grounded`; a pure routing function then decides between finalizing and one regeneration.
The cap (original attempt + 1 retry) keeps the loop bounded. The retry reuses the same prompt and context, and the factory's
default temperature is 0, so a retry often reproduces a similar answer — the cap, not the retry, is what guarantees termination.

In [ ]:
# ============================================================================
# Node 4 — Hallucination Check + Routing Function
# ============================================================================

class HallucinationGrade(BaseModel):
    """Grade whether a response is grounded in the retrieved documents."""
    grade: Literal["grounded", "not_grounded"] = Field(
        description="'grounded' if the response is fully supported by the documents, 'not_grounded' otherwise."
    )


hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a hallucination grader.
Given a set of retrieved documents and a generated response, determine whether
the response is fully supported by the documents.

Return 'grounded' if every claim in the response can be traced back to the
provided documents. Return 'not_grounded' if the response contains information
that is not present in or contradicts the documents."""),
    ("human", """Retrieved Documents:
{documents}

Generated Response:
{response}

Is this response grounded in the documents?"""),
])

hallucination_chain = hallucination_prompt | llm.with_structured_output(HallucinationGrade)


def hallucination_check(state: AgentState) -> dict:
    """Grade whether the response is grounded in retrieved docs."""
    result = hallucination_chain.invoke({
        "documents": state["retrieved_docs"],
        "response": state["response"],
    })
    retries = state.get("retries", 0)
    print(f"[hallucination_check] Grade: {result.grade}  (retries so far: {retries})")
    return {
        "hallucination_grade": result.grade,
        "retries": retries + 1,
    }


def route_after_hallucination_check(state: AgentState) -> str:
    """If grounded or max retries reached → finalize, else → regenerate."""
    if state["hallucination_grade"] == "grounded":
        return "finalize"
    if state.get("retries", 0) >= 2:       # original attempt + 1 retry
        print("[router] Max retries reached — ending with current response.")
        return "finalize"
    print("[router] Not grounded — regenerating...")
    return "regenerate"

### 2.6 🧷 Node 5 — Finalize

Only a finished turn is written to history: the user query and the accepted answer.

In [ ]:
# ============================================================================
# Node 5 — Finalize (Record Conversation Turn)
# ============================================================================

def finalize(state: AgentState) -> dict:
    """Append the user query and AI response as messages for conversation history."""
    return {
        "messages": [
            HumanMessage(content=state["query"]),
            AIMessage(content=state["response"]),
        ]
    }

---
## 🔁 Part 3: Build the Graph with Memory

Four straight edges and one conditional edge make the pipeline; compiling with `MemorySaver` stores each conversation
under its `thread_id`. The `chat()` helper uses the user id as the thread id, which is what isolates users from each other.

In [ ]:
# ============================================================================
# Build & Compile the LangGraph (with Memory)
# ============================================================================

workflow = StateGraph(AgentState)

# --- Nodes ---
workflow.add_node("categorize_query", categorize_query)
workflow.add_node("retrieve_docs", retrieve_docs)
workflow.add_node("generate_response", generate_response)
workflow.add_node("hallucination_check", hallucination_check)
workflow.add_node("finalize", finalize)

# --- Edges ---
workflow.add_edge(START, "categorize_query")
workflow.add_edge("categorize_query", "retrieve_docs")
workflow.add_edge("retrieve_docs", "generate_response")
workflow.add_edge("generate_response", "hallucination_check")

workflow.add_conditional_edges(
    "hallucination_check",
    route_after_hallucination_check,
    {
        "finalize": "finalize",
        "regenerate": "generate_response",
    },
)

workflow.add_edge("finalize", END)

# --- Compile with MemorySaver for per-user conversation persistence ---
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)

In [ ]:
# ============================================================================
# Graph Visualization
# ============================================================================

from IPython.display import Image, Markdown, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    display(Markdown(f"```mermaid\n{graph.get_graph().draw_mermaid()}\n```"))

In [ ]:
# ============================================================================
# Chat Helper — Multi-User Convenience Wrapper
# ============================================================================

def chat(query: str, user_id: str) -> str:
    """
    Send a query to the ShopUNow agent as a specific user.

    Conversation history is automatically maintained per user_id
    via LangGraph's MemorySaver checkpointer.
    """
    result = graph.invoke(
        {
            "query": query,
            "department": None,
            "retrieved_docs": None,
            "response": None,
            "hallucination_grade": None,
            "retries": 0,
        },
        config={"configurable": {"thread_id": user_id}},
    )
    return result["response"]

---
## 👥 Part 4: Multi-User Conversational Demo

Below we demonstrate two independent users interacting with the system. Each user has **isolated conversation history**, and follow-up queries correctly reference prior context.

In [ ]:
# ============================================================================
# Demo — User A (employee_42): Multi-turn HR → IT conversation
# ============================================================================
from IPython.display import display, Markdown

USER_A = "employee_42"

# Turn 1: Initial HR question
print("=" * 80)
print(f"[{USER_A}] Turn 1: Initial HR question")
print("=" * 80)
response = chat("How many paid leaves do I get per year?", USER_A)
display(Markdown(response))

# Turn 2: Follow-up — should stay in HR context
print("\n" + "=" * 80)
print(f"[{USER_A}] Turn 2: Follow-up (should stay in HR)")
print("=" * 80)
response = chat("What about carry-forward? Can I carry unused leaves to next year?", USER_A)
display(Markdown(response))

# Turn 3: Topic switch to IT support
print("\n" + "=" * 80)
print(f"[{USER_A}] Turn 3: Topic switch to IT support")
print("=" * 80)
response = chat("I also need help resetting my VPN password.", USER_A)
display(Markdown(response))

In [ ]:
# ============================================================================
# Demo — User B (customer_99): Separate customer service session
# ============================================================================

USER_B = "customer_99"

# Turn 1: Customer service question
print("=" * 80)
print(f"[{USER_B}] Turn 1: Damaged product report")
print("=" * 80)
response = chat("I received a damaged product, what should I do?", USER_B)
display(Markdown(response))

# Turn 2: Follow-up about refund timeline
print("\n" + "=" * 80)
print(f"[{USER_B}] Turn 2: Follow-up about refund")
print("=" * 80)
response = chat("How long will the refund take?", USER_B)
display(Markdown(response))

# ---- Isolation check ----
print("\n" + "=" * 80)
print("ISOLATION CHECK")
print("=" * 80)
state_a = graph.get_state({"configurable": {"thread_id": USER_A}})
state_b = graph.get_state({"configurable": {"thread_id": USER_B}})
msgs_a = state_a.values.get("messages", [])
msgs_b = state_b.values.get("messages", [])
print(f"User A ({USER_A}) message count: {len(msgs_a)} ({len(msgs_a)//2} turns)")
print(f"User B ({USER_B}) message count: {len(msgs_b)} ({len(msgs_b)//2} turns)")
# Isolated means no message of one user appears in the other user's history
shared = {m.content for m in msgs_a} & {m.content for m in msgs_b}
print("Histories are isolated!" if not shared else f"WARNING: {len(shared)} message(s) appear in both histories!")

Read the `[categorize_query]` lines: User A's carry-forward follow-up should stay in `hr` and the VPN question should switch to `it_support`.
The isolation check expects 3 turns for User A, 2 for User B, and no message shared between them.

---
## 🧪 Part 5: Negative Test Cases

These tests verify the system handles **adversarial and out-of-scope inputs** gracefully:

1. **Completely irrelevant query** — a question no department can answer (e.g., astrophysics)
2. **Gibberish input** — nonsensical string with no semantic meaning
3. **Cross-user leakage probe** — a new user tries to reference another user's conversation

The pass/fail lines are keyword heuristics on the response text; read each response as well.

In [ ]:
# ============================================================================
# Negative Test Cases
# ============================================================================

NEGATIVE_USER = "negative_tester"

negative_cases = [
    {
        "name": "Irrelevant query (out-of-scope topic)",
        "query": "What is the escape velocity of Mars?",
        "expect": "The response should NOT fabricate a ShopUNow policy. "
                  "It should indicate the information is unavailable in the knowledge base.",
    },
    {
        "name": "Gibberish / nonsensical input",
        "query": "asdkjh 1234 zxcvb !@#$% qwerty",
        "expect": "The system should still run without errors. "
                  "Response may indicate it cannot understand the query.",
    },
    {
        "name": "Cross-user leakage probe",
        "query": "What did employee_42 ask about in their last conversation?",
        "expect": "The system must NOT leak another user's conversation history. "
                  "It should have no knowledge of employee_42's session.",
    },
]

for i, case in enumerate(negative_cases, 1):
    print("=" * 80)
    print(f"NEGATIVE TEST {i}: {case['name']}")
    print(f"  Query   : {case['query'][:80]}{'...' if len(case['query']) > 80 else ''}")
    print(f"  Expected: {case['expect']}")
    print("=" * 80)

    response = chat(case["query"], NEGATIVE_USER)
    display(Markdown(f"**Response:**\n\n{response}"))

    response_lower = response.lower()
    if case["name"].startswith("Irrelevant"):
        passed = any(
            phrase in response_lower
            for phrase in [
                "not available", "don't have", "do not have",
                "no information", "outside", "cannot", "can't",
                "not in", "doesn't contain", "does not contain",
                "not enough information",
            ]
        )
    elif case["name"].startswith("Gibberish"):
        passed = response is not None and len(response) > 0
    elif case["name"].startswith("Cross-user"):
        passed = "paid leave" not in response_lower and "vpn" not in response_lower
    else:
        passed = True

    status = "✅ PASSED" if passed else "❌ FAILED"
    print(f"\n  Result: {status}\n")

---
## 💾 Part 6 (Optional): Persistent Storage with SqliteSaver

The `MemorySaver` above stores conversation history **in memory** — it's lost when the kernel restarts. For production use, swap to `SqliteSaver` to persist conversations to a SQLite database file.

The cell below is commented out; uncomment it to try it. It writes `memory.db` in this folder.

In [ ]:
# ============================================================================
# Persistent Memory with SqliteSaver
# ============================================================================
# Uncomment and run this cell to use persistent storage instead of MemorySaver.
# Conversations will survive kernel restarts.

# from langgraph.checkpoint.sqlite import SqliteSaver
#
# # Build the same workflow
# persistent_workflow = StateGraph(AgentState)
# persistent_workflow.add_node("categorize_query", categorize_query)
# persistent_workflow.add_node("retrieve_docs", retrieve_docs)
# persistent_workflow.add_node("generate_response", generate_response)
# persistent_workflow.add_node("hallucination_check", hallucination_check)
# persistent_workflow.add_node("finalize", finalize)
#
# persistent_workflow.add_edge(START, "categorize_query")
# persistent_workflow.add_edge("categorize_query", "retrieve_docs")
# persistent_workflow.add_edge("retrieve_docs", "generate_response")
# persistent_workflow.add_edge("generate_response", "hallucination_check")
# persistent_workflow.add_conditional_edges(
#     "hallucination_check",
#     route_after_hallucination_check,
#     {"finalize": "finalize", "regenerate": "generate_response"},
# )
# persistent_workflow.add_edge("finalize", END)
#
# # Compile with SqliteSaver — data persists to memory.db
# with SqliteSaver.from_conn_string("memory.db") as persistent_memory:
#     persistent_graph = persistent_workflow.compile(checkpointer=persistent_memory)
#
#     # Example usage — this conversation survives kernel restarts
#     result = persistent_graph.invoke(
#         {
#             "query": "What is the return policy?",
#             "department": None,
#             "retrieved_docs": None,
#             "response": None,
#             "hallucination_grade": None,
#             "retries": 0,
#         },
#         config={"configurable": {"thread_id": "persistent_user_1"}},
#     )
#     print(result["response"][:300])

---
## 📝 Summary

In this notebook, we learned:

### 1. Department vector stores
- The agent reads one persisted Chroma store per department and rebuilds a missing one from `data/`, with the same embedding model that built the others.

### 2. Graph state and nodes
- Classification and grading use structured output, so routing decisions are typed values, not parsed text.
- Classifier and generator both see trimmed recent history, which is what lets follow-up questions resolve.

### 3. Bounded self-correction with memory
- The hallucination grade feeds a pure routing function with a hard cap: at most one regeneration, and the graph always returns an answer.
- `MemorySaver` keyed by `thread_id` gives each user their own conversation history.

### 4. Multi-user conversations
- Follow-ups stay in the right department, topic switches re-route, and two users' histories share nothing.

### 5. Negative tests
- Out-of-scope and gibberish inputs should produce an "information not available" answer, not an invented policy; a new user cannot see another user's history.

### 6. Persistent memory (optional)
- `SqliteSaver` is a drop-in replacement for `MemorySaver` when conversations must survive restarts.

### Next Steps
- **`WALKTHROUGH.md`** (in this project's folder): a node-by-node explanation of the design and its trade-offs.
- Extend the agent: add a department, an escalation path for `not_grounded` answers after the retry, or a web-search fallback.